In [ ]:
!pip install -q librosa soundfile tqdm scikit-learn matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os
import librosa
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, log_loss


Mounted at /content/drive


In [ ]:
def extract_features(file_path, sr=22050, duration=40):
    try:
        y, sr = librosa.load(file_path, sr=sr, duration=duration)
        if len(y) == 0:
            return None

        mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).T, axis=0)
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr).T, axis=0)
        mel = np.mean(librosa.feature.melspectrogram(y=y, sr=sr).T, axis=0)
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        spectral_rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

        # Combine all features into one vector
        feature_vector = np.hstack([mfcc, chroma, mel[:60], [zcr, spectral_centroid, spectral_rolloff]])
        return feature_vector
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/ISBInternshipAIML/datasets"

features, genres, languages, groups = [], [], [], []
for genre in os.listdir(DATASET_PATH):
    genre_path = os.path.join(DATASET_PATH, genre)
    if not os.path.isdir(genre_path):
        continue

    for lang in os.listdir(genre_path):
        lang_path = os.path.join(genre_path, lang)
        if not os.path.isdir(lang_path):
            continue

        for file in tqdm(os.listdir(lang_path), desc=f"{genre}/{lang}"):
            if not file.endswith(".mp3"):
                continue
            file_path = os.path.join(lang_path, file)
            feat = extract_features(file_path)
            if feat is not None:
                features.append(feat)
                genres.append(genre)
                languages.append(lang)
                groups.append(file)   # unique file acts as group


EDM/eng:  75%|███████▌  | 154/204 [01:38<00:31,  1.58it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
Classical/kannada: 100%|██████████| 22/22 [01:11<00:00,  3.23s/it]


In [ ]:
X = np.array(features)
print("Feature matrix:", X.shape)


Feature matrix: (1211, 95)


In [ ]:
genre_enc = LabelEncoder()
lang_enc = LabelEncoder()

y_genre = genre_enc.fit_transform(genres)
y_lang = lang_enc.fit_transform(languages)

print("Genres:", genre_enc.classes_)
print("Languages:", lang_enc.classes_)


Genres: ['Classical' 'Devotional' 'EDM' 'Folk' 'Hip-Hop' 'Pop' 'Rock' 'film']
Languages: ['Bengali devotional songs' 'Bhojpuri devotional songs' 'English'
 'Gujarati' 'Gujarati devotional songs' 'Hindi' 'Hindi devotional songs'
 'Korean' 'Malayalam' 'Marathi devotional songs' 'Punjabi'
 'Punjabi devotional songs' 'Rajasthani' 'Spanish' 'Tamil' 'Telugu' 'eng'
 'hindi' 'kannada' 'tamil devotional songs' 'telugu devotional songs']


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def group_split(X, y, groups, test_size=0.2):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    for train_idx, test_idx in gss.split(X, y, groups):
        return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


In [ ]:
X_train_g, X_test_g, y_train_g, y_test_g = group_split(X, y_genre, groups)
X_train_l, X_test_l, y_train_l, y_test_l = group_split(X, y_lang, groups)


In [ ]:
genre_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

lang_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)


In [ ]:
print("\nTraining Genre Classifier...")
genre_model.fit(X_train_g, y_train_g)

print("\nTraining Language Classifier...")
lang_model.fit(X_train_l, y_train_l)



Training Genre Classifier...

Training Language Classifier...


RandomForestClassifier(max_depth=15, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, label_enc, title):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)

    y_prob_train = model.predict_proba(X_train)
    y_prob_test = model.predict_proba(X_test)

    # Use full label range to handle missing classes in test set
    all_labels = np.arange(len(label_enc.classes_))

    train_loss = log_loss(y_train, y_prob_train, labels=all_labels)
    test_loss = log_loss(y_test, y_prob_test, labels=all_labels)

    print(f"\n========== {title} ==========")
    print(f"Train Accuracy: {train_acc*100:.2f}% | Train Loss: {train_loss:.4f}")
    print(f"Test Accuracy:  {test_acc*100:.2f}% | Test Loss:  {test_loss:.4f}\n")

    print("Classification Report:")
    print(classification_report(
        y_test, y_pred_test,
        target_names=label_enc.classes_,
        zero_division=0  # avoid warnings for empty predictions
    ))

    return train_acc, test_acc, train_loss, test_loss


In [ ]:
genre_metrics = evaluate_model(genre_model, X_train_g, y_train_g, X_test_g, y_test_g, genre_enc, "Genre Classifier")
lang_metrics = evaluate_model(lang_model, X_train_l, y_train_l, X_test_l, y_test_l, lang_enc, "Language Classifier")



========== Genre Classifier ==========
Train Accuracy: 99.90% | Train Loss: 0.3108
Test Accuracy:  45.87% | Test Loss:  1.5107

Classification Report:
              precision    recall  f1-score   support

   Classical       1.00      0.12      0.22         8
  Devotional       0.17      0.05      0.08        20
         EDM       0.51      0.78      0.62        46
        Folk       0.36      0.57      0.44        44
     Hip-Hop       0.67      0.61      0.64        46
         Pop       0.45      0.26      0.33        39
        Rock       0.00      0.00      0.00         5
        film       0.32      0.29      0.31        34

    accuracy                           0.46       242
   macro avg       0.44      0.34      0.33       242
weighted avg       0.45      0.46      0.43       242


========== Language Classifier ==========
Train Accuracy: 99.90% | Train Loss: 0.3910
Test Accuracy:  39.67% | Test Loss:  2.4794

Classification Report:


ValueError: Number of classes, 19, does not match size of target_names, 21. Try specifying the labels parameter